In [2]:
import os
import openai

from dotenv import load_dotenv

_ = load_dotenv()

## Basic call - no function calling yet

In [3]:
#any message you pass to an LLM has to be strictly in this format:
messages = [
    {
        "role": "user",
        "content": "Hi"
    }
]

In [4]:
response = openai.ChatCompletion.create(
    model = "gpt-3.5-turbo",
    messages = messages,
)

In [5]:
response["choices"][0]["message"]["content"]

'Hello! How are you today?'

## Defining a function and giving the LLM access to it

In [6]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be a backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

In [7]:
# define a function
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
] # function defination should always be in this format

In [8]:
messages = [
    {
        "role": "user",
        "content": "What is the weather in San Francisco, CA?"

    }
]

In [9]:
response = openai.ChatCompletion.create(
    model = "gpt-3.5-turbo",
    messages = messages,
    functions = functions
)

In [10]:
response["choices"][0]

<OpenAIObject at 0x7f0acaa70680> JSON: {
  "index": 0,
  "message": {
    "role": "assistant",
    "content": null,
    "function_call": {
      "name": "get_current_weather",
      "arguments": "{\"location\":\"San Francisco, CA\"}"
    },
    "refusal": null,
    "annotations": []
  },
  "logprobs": null,
  "finish_reason": "function_call"
}

You can see the LLM extracting `"San Francisco, CA"` as the argument to pass into the function.

In [11]:
args = json.loads(response["choices"][0]["message"]["function_call"]["arguments"])

In [12]:
get_current_weather(**args) #need ** to unpack the dict into keyword arguments - get_current_weather(args) would incorrectly pass the whole dict as the `location` argument


'{"location": {"location": "San Francisco, CA"}, "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'

## Forcing function calls

You can use the `function_call` parameter to control whether (and how) the LLM calls a function. There are three options:
1. `function_call="auto"` (default) - lets the LLM decide whether to call a function or not
2. `function_call="none"` - forces the LLM to not call any function
3. `function_call={"name": func_name}` - forces the LLM to call that specific function


In [13]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

In [14]:
response = openai.ChatCompletion.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call = {"name": "get_current_weather"}
)

response

<OpenAIObject chat.completion id=chatcmpl-BUsxEG1eTCxlzbVhINBuHe30anTlZ at 0x7f0acaa70e50> JSON: {
  "id": "chatcmpl-BUsxEG1eTCxlzbVhINBuHe30anTlZ",
  "object": "chat.completion",
  "created": 1746700784,
  "model": "gpt-3.5-turbo-0125",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\"location\":\"San Francisco, CA\"}"
        },
        "refusal": null,
        "annotations": []
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 86,
    "completion_tokens": 9,
    "total_tokens": 95,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "service_tier": "default",


If you comment out the `functions` parameter, you'll see fewer tokens being used - that's because passing `functions` also sends the LLM the full function schema/description as part of the prompt, which costs tokens. Worth keeping in mind if you're working with a tight token budget.

## Full loop: ask -> call the function -> feed the result back to the LLM

Now we'll ask the LLM about the weather, actually call the function with the arguments it gives us, and then feed the function's output back into the LLM so it can generate a natural language response.


In [16]:
messages = [
    {
        "role": "user",
        "content": "what is the current weather in Boston"
    }
]

response = openai.ChatCompletion.create(
    model = "gpt-3.5-turbo",
    messages = messages,
    functions = functions
)

response

<OpenAIObject chat.completion id=chatcmpl-EEWy0Wb12qVgsbwkPH7dZIj1zJaKI at 0x7f0b0bdb2ef0> JSON: {
  "id": "chatcmpl-EEWy0Wb12qVgsbwkPH7dZIj1zJaKI",
  "object": "chat.completion",
  "created": 1787132024,
  "model": "gpt-3.5-turbo-0125",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\"location\":\"Boston\",\"unit\":\"celsius\"}"
        },
        "refusal": null,
        "annotations": []
      },
      "logprobs": null,
      "finish_reason": "function_call"
    }
  ],
  "usage": {
    "prompt_tokens": 81,
    "completion_tokens": 20,
    "total_tokens": 101,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "servi

In [17]:
messages.append(response["choices"][0]["message"]) #yes, exactly that - this adds the llm's own message (which includes its decision to call get_current_weather with certain args) into the conversation history, so that context isn't lost. next we'll also append the function's actual output as a "function" role message, so the llm has both its own request AND the result when generating the final reply


In [19]:
args = json.loads(response["choices"][0]["message"]["function_call"]["arguments"])
observation = get_current_weather(args)

In [20]:
messages.append(
    {
        "role": "function", #the "function" role tells the llm this message is the output of a function call, not a human or the llm itself
        "name": "get_current_weather",
        "content": observation
    }
)


In [22]:
messages #final messages that are being passed to the llm

[{'role': 'user', 'content': 'what is the current weather in Boston'},
 <OpenAIObject at 0x7f0acaa84450> JSON: {
   "role": "assistant",
   "content": null,
   "function_call": {
     "name": "get_current_weather",
     "arguments": "{\"location\":\"Boston\",\"unit\":\"celsius\"}"
   },
   "refusal": null,
   "annotations": []
 },
 {'role': 'function',
  'name': 'get_current_weather',
  'content': '{"location": {"location": "Boston", "unit": "celsius"}, "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'}]

In [21]:
response = openai.ChatCompletion.create(
    model = "gpt-3.5-turbo",
    messages = messages,
    functions = functions
)

response

<OpenAIObject chat.completion id=chatcmpl-EEX59eeYcTkylRpxUjeRSldrhi1dj at 0x7f0acaa70d60> JSON: {
  "id": "chatcmpl-EEX59eeYcTkylRpxUjeRSldrhi1dj",
  "object": "chat.completion",
  "created": 1787132467,
  "model": "gpt-3.5-turbo-0125",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The current weather in Boston is 72\u00b0F with sunny and windy conditions.",
        "refusal": null,
        "annotations": []
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 150,
    "completion_tokens": 16,
    "total_tokens": 166,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "service_tier": "default",
  "system_fingerprint": null
}